# Aula 3 - Mastering Machine Learning Advanced

## Séries Temporais: ARIMA, Prophet e LSTM

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

## Índice

1. [Introdução](#1)
2. [Criando o Dataset de Telecom](#2)
3. [Componentes de uma Série Temporal](#3)
4. [Estacionariedade e Teste ADF](#4)
5. [ARIMA](#5)
   - 5.1 [Identificando os Parâmetros p, d, q](#51)
   - 5.2 [Treinando e Avaliando o ARIMA](#52)
6. [Prophet](#6)
7. [LSTM para Séries Temporais](#7)
8. [Comparativo Final](#8)
9. [Conclusão](#9)

# 1. Introdução <a id="1"></a>

Uma **série temporal** é uma sequência de observações coletadas ao longo do tempo em intervalos regulares. Diferente dos datasets que vimos até agora, aqui a **ordem dos dados importa** — a observação de hoje depende das observações anteriores.

**Por que isso importa em Telecom?**

Operadoras de telecomunicações precisam responder perguntas como:
- Quantos assinantes teremos no próximo trimestre?
- Qual será a receita mensal recorrente (MRR) nos próximos 6 meses?
- Há um padrão sazonal no consumo de dados que precisamos antecipar?

Nesta aula vamos cobrir três abordagens complementares:

| Abordagem | Quando usar |
|---|---|
| **ARIMA** | Série estacionária, poucos dados, explicabilidade necessária |
| **Prophet** | Séries com sazonalidade forte, feriados, fácil de usar |
| **LSTM** | Séries longas, padrões complexos, quando mais dados estão disponíveis |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

np.random.seed(42)

# 2. Criando o Dataset de Telecom <a id="2"></a>

Vamos simular 5 anos de dados mensais de uma operadora de telecomunicações: receita mensal recorrente (MRR) e base de assinantes ativos. O dataset incorpora tendência de crescimento, sazonalidade anual e ruído aleatório — padrões reais de uma operadora.

In [ ]:
# gerando 60 meses (jan/2019 a dez/2023)
datas = pd.date_range(start='2019-01-01', periods=60, freq='MS')
t = np.arange(60)

# receita mensal: tendência linear + sazonalidade anual + ruído
tendencia    = 5000 + 80 * t
sazonalidade = 800 * np.sin(2 * np.pi * t / 12) + 400 * np.cos(2 * np.pi * t / 6)
ruido        = np.random.normal(0, 300, 60)
mrr          = tendencia + sazonalidade + ruido

# assinantes ativos: mesma estrutura, escala diferente
assinantes = (1200 + 15 * t
              + 120 * np.sin(2 * np.pi * t / 12)
              + np.random.normal(0, 40, 60)).astype(int)

df_ts = pd.DataFrame({
    'data': datas,
    'mrr': mrr.round(2),
    'assinantes': assinantes
}).set_index('data')

df_ts.head(12)

In [ ]:
# visualizando as duas séries
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(df_ts.index, df_ts['mrr'], color='steelblue', linewidth=1.8)
axes[0].fill_between(df_ts.index, df_ts['mrr'], alpha=0.15, color='steelblue')
axes[0].set_title('Receita Mensal Recorrente (MRR) — Operadora Telecom')
axes[0].set_ylabel('MRR (R$)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

axes[1].plot(df_ts.index, df_ts['assinantes'], color='coral', linewidth=1.8)
axes[1].fill_between(df_ts.index, df_ts['assinantes'], alpha=0.15, color='coral')
axes[1].set_title('Assinantes Ativos')
axes[1].set_ylabel('Assinantes')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

plt.tight_layout()
plt.show()

# 3. Componentes de uma Série Temporal <a id="3"></a>

Toda série temporal pode ser decomposta em quatro componentes:

- **Tendência**: direção de longo prazo (crescimento, queda ou estável)
- **Sazonalidade**: padrões que se repetem em intervalos fixos
- **Ciclo**: flutuações de longo prazo sem período fixo (ex: ciclos econômicos)
- **Resíduo**: ruído aleatório que não é explicado pelos demais

In [ ]:
# decomposição aditiva da série de MRR
decomposicao = seasonal_decompose(df_ts['mrr'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

componentes = [
    (df_ts['mrr'],              'Série Original',  'steelblue'),
    (decomposicao.trend,        'Tendência',        'seagreen'),
    (decomposicao.seasonal,     'Sazonalidade',     'coral'),
    (decomposicao.resid,        'Resíduo',          'mediumpurple'),
]

for ax, (dados, titulo, cor) in zip(axes, componentes):
    ax.plot(dados, color=cor, linewidth=1.5)
    ax.set_title(titulo)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))

plt.suptitle('Decomposição da Série Temporal — MRR', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# 4. Estacionariedade e Teste ADF <a id="4"></a>

Uma série é **estacionária** quando suas propriedades estatísticas (média, variância) não mudam ao longo do tempo. O ARIMA exige que a série seja estacionária.

O **Teste ADF (Augmented Dickey-Fuller)** testa formalmente a estacionariedade:
- **H₀**: a série tem raiz unitária (não estacionária)
- **p-value < 0.05**: rejeitamos H₀ → série estacionária

In [ ]:
def teste_adf(serie, nome):
    resultado = adfuller(serie.dropna())
    print(f'--- Teste ADF: {nome} ---')
    print(f'  Estatística ADF: {resultado[0]:.4f}')
    print(f'  p-value:         {resultado[1]:.4f}')
    print(f'  Valores críticos: {resultado[4]}')
    if resultado[1] < 0.05:
        print('  → SÉRIE ESTACIONÁRIA (rejeita H₀)\n')
    else:
        print('  → SÉRIE NÃO ESTACIONÁRIA (não rejeita H₀)\n')

teste_adf(df_ts['mrr'], 'MRR original')

In [ ]:
# aplicando diferenciação de primeira ordem para tornar a série estacionária
mrr_diff = df_ts['mrr'].diff().dropna()

teste_adf(mrr_diff, 'MRR diferenciada (d=1)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df_ts['mrr'], color='steelblue')
axes[0].set_title('MRR Original')
axes[1].plot(mrr_diff, color='coral')
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].set_title('MRR Diferenciada (d=1)')
plt.tight_layout()
plt.show()

# 5. ARIMA <a id="5"></a>

O **ARIMA(p, d, q)** combina três componentes:

- **AR(p)**: AutoRegressivo — usa os *p* valores anteriores para prever o próximo
- **I(d)**: Integrado — número de diferenciações para tornar a série estacionária
- **MA(q)**: Média Móvel — usa os *q* erros anteriores na previsão

## 5.1 Identificando os Parâmetros p, d, q <a id="51"></a>

- **d**: determinado pelo número de diferenciações necessárias (vimos que d=1)
- **p**: identificado pelo gráfico PACF (Partial Autocorrelation)
- **q**: identificado pelo gráfico ACF (Autocorrelation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(mrr_diff, lags=24, ax=axes[0], color='steelblue')
axes[0].set_title('ACF — MRR Diferenciada (identifica q)')

plot_pacf(mrr_diff, lags=24, ax=axes[1], color='coral')
axes[1].set_title('PACF — MRR Diferenciada (identifica p)')

plt.tight_layout()
plt.show()

## 5.2 Treinando e Avaliando o ARIMA <a id="52"></a>

In [ ]:
# separando treino e teste: últimos 12 meses como teste
treino = df_ts['mrr'][:-12]
teste  = df_ts['mrr'][-12:]

# treinando ARIMA(1,1,1)
modelo_arima = ARIMA(treino, order=(1, 1, 1))
resultado_arima = modelo_arima.fit()

print(resultado_arima.summary())

In [ ]:
# prevendo os próximos 12 meses
previsao_arima = resultado_arima.forecast(steps=12)
previsao_arima.index = teste.index

# métricas
mae_arima  = mean_absolute_error(teste, previsao_arima)
rmse_arima = mean_squared_error(teste, previsao_arima) ** 0.5
mape_arima = (np.abs((teste - previsao_arima) / teste)).mean() * 100

print(f'ARIMA(1,1,1) — MAE: {mae_arima:.2f} | RMSE: {rmse_arima:.2f} | MAPE: {mape_arima:.2f}%')

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(treino, label='Treino', color='steelblue')
plt.plot(teste,  label='Real (teste)', color='seagreen')
plt.plot(previsao_arima, label='Previsão ARIMA', color='coral', linestyle='--', marker='o', markersize=4)
plt.axvline(teste.index[0], color='gray', linestyle=':', label='Início da previsão')
plt.title('ARIMA(1,1,1) — Previsão de MRR')
plt.ylabel('MRR (R$)')
plt.legend()
plt.tight_layout()
plt.show()

# 6. Prophet <a id="6"></a>

O **Prophet** foi desenvolvido pelo time de Data Science do Meta (Facebook) para forecasting em larga escala. Seus diferenciais:

- Lida bem com **sazonalidade múltipla** (anual, mensal, semanal)
- Detecta **pontos de mudança** (structural breaks) automaticamente
- Incorpora **feriados e eventos especiais**
- Interface simples, poucos hiperparâmetros para ajustar

In [ ]:
!pip install prophet -q

In [ ]:
from prophet import Prophet

# Prophet exige colunas 'ds' (data) e 'y' (valor)
df_prophet = df_ts['mrr'].reset_index()
df_prophet.columns = ['ds', 'y']

treino_prophet = df_prophet[:-12]
teste_prophet  = df_prophet[-12:]

# treinando o modelo
modelo_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='additive'
)
modelo_prophet.fit(treino_prophet)

In [ ]:
# gerando previsões para os próximos 12 meses
futuro = modelo_prophet.make_future_dataframe(periods=12, freq='MS')
previsao_prophet = modelo_prophet.predict(futuro)

# extraindo apenas o período de teste
prev_teste_prophet = previsao_prophet[previsao_prophet['ds'].isin(teste_prophet['ds'])]

mae_prophet  = mean_absolute_error(teste_prophet['y'].values, prev_teste_prophet['yhat'].values)
rmse_prophet = mean_squared_error(teste_prophet['y'].values, prev_teste_prophet['yhat'].values) ** 0.5
mape_prophet = (np.abs((teste_prophet['y'].values - prev_teste_prophet['yhat'].values) / teste_prophet['y'].values)).mean() * 100

print(f'Prophet — MAE: {mae_prophet:.2f} | RMSE: {rmse_prophet:.2f} | MAPE: {mape_prophet:.2f}%')

In [ ]:
# visualizando a previsão do Prophet com intervalos de confiança
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(df_prophet['ds'], df_prophet['y'], label='Real', color='steelblue')
ax.plot(prev_teste_prophet['ds'], prev_teste_prophet['yhat'],
        label='Previsão Prophet', color='coral', linestyle='--', marker='o', markersize=4)
ax.fill_between(
    prev_teste_prophet['ds'],
    prev_teste_prophet['yhat_lower'],
    prev_teste_prophet['yhat_upper'],
    alpha=0.2, color='coral', label='Intervalo de Confiança'
)
ax.axvline(teste_prophet['ds'].iloc[0], color='gray', linestyle=':', label='Início da previsão')
ax.set_title('Prophet — Previsão de MRR com Intervalo de Confiança')
ax.set_ylabel('MRR (R$)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# decomposição automática dos componentes pelo Prophet
fig2 = modelo_prophet.plot_components(previsao_prophet)
plt.suptitle('Componentes da Previsão — Prophet', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# 7. LSTM para Séries Temporais <a id="7"></a>

As redes **LSTM (Long Short-Term Memory)** são uma arquitetura de redes neurais recorrentes especialmente projetada para aprender dependências de longo prazo em sequências. Em séries temporais, cada passo da sequência alimenta a memória da rede, que decide o que preservar e o que esquecer.

A abordagem LSTM é mais poderosa para padrões complexos, mas exige mais dados e mais cuidado no pré-processamento.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow: {tf.__version__}')

In [ ]:
# normalizando os dados — LSTM é sensível à escala
serie = df_ts['mrr'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
serie_norm = scaler.fit_transform(serie)

# criando janelas deslizantes: usamos 'n_passos' meses para prever o próximo
def criar_janelas(serie, n_passos):
    X, y = [], []
    for i in range(n_passos, len(serie)):
        X.append(serie[i - n_passos:i, 0])
        y.append(serie[i, 0])
    return np.array(X), np.array(y)

N_PASSOS = 12  # usa os últimos 12 meses para prever o próximo
X_lstm, y_lstm = criar_janelas(serie_norm, N_PASSOS)

# separando treino e teste (últimos 12 meses)
split = len(X_lstm) - 12
X_train_lstm, X_test_lstm = X_lstm[:split], X_lstm[split:]
y_train_lstm, y_test_lstm = y_lstm[:split], y_lstm[split:]

# reshape para (amostras, passos de tempo, features)
X_train_lstm = X_train_lstm.reshape(-1, N_PASSOS, 1)
X_test_lstm  = X_test_lstm.reshape(-1, N_PASSOS, 1)

print(f'X_train: {X_train_lstm.shape} | X_test: {X_test_lstm.shape}')

In [ ]:
# construindo o modelo LSTM
tf.random.set_seed(42)

modelo_lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(N_PASSOS, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

modelo_lstm.compile(optimizer='adam', loss='mse')
modelo_lstm.summary()

In [ ]:
# treinando com early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

historico = modelo_lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=100,
    batch_size=8,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0
)

# curva de perda
plt.figure(figsize=(9, 4))
plt.plot(historico.history['loss'],     label='Treino',    color='steelblue')
plt.plot(historico.history['val_loss'], label='Validação', color='coral')
plt.title('LSTM — Curva de Perda (MSE)')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# prevendo e revertendo a normalização
prev_lstm_norm = modelo_lstm.predict(X_test_lstm)
prev_lstm = scaler.inverse_transform(prev_lstm_norm).flatten()
real_lstm = scaler.inverse_transform(y_test_lstm.reshape(-1, 1)).flatten()

mae_lstm  = mean_absolute_error(real_lstm, prev_lstm)
rmse_lstm = mean_squared_error(real_lstm, prev_lstm) ** 0.5
mape_lstm = (np.abs((real_lstm - prev_lstm) / real_lstm)).mean() * 100

print(f'LSTM — MAE: {mae_lstm:.2f} | RMSE: {rmse_lstm:.2f} | MAPE: {mape_lstm:.2f}%')

In [ ]:
# visualizando previsão do LSTM
datas_teste = df_ts.index[-12:]

plt.figure(figsize=(13, 5))
plt.plot(df_ts.index, df_ts['mrr'], label='Real', color='steelblue')
plt.plot(datas_teste, prev_lstm, label='Previsão LSTM',
         color='mediumpurple', linestyle='--', marker='o', markersize=4)
plt.axvline(datas_teste[0], color='gray', linestyle=':', label='Início da previsão')
plt.title('LSTM — Previsão de MRR')
plt.ylabel('MRR (R$)')
plt.legend()
plt.tight_layout()
plt.show()

# 8. Comparativo Final <a id="8"></a>

In [ ]:
# tabela comparativa das três abordagens
df_comparativo = pd.DataFrame([
    {'Modelo': 'ARIMA(1,1,1)', 'MAE': mae_arima,  'RMSE': rmse_arima,  'MAPE (%)': mape_arima},
    {'Modelo': 'Prophet',      'MAE': mae_prophet, 'RMSE': rmse_prophet, 'MAPE (%)': mape_prophet},
    {'Modelo': 'LSTM',         'MAE': mae_lstm,    'RMSE': rmse_lstm,   'MAPE (%)': mape_lstm},
]).set_index('Modelo').round(2)

df_comparativo.sort_values('MAPE (%)')

In [ ]:
# comparativo visual: real vs. previsões dos três modelos
plt.figure(figsize=(13, 6))
plt.plot(df_ts.index, df_ts['mrr'], label='Real', color='steelblue', linewidth=2)

prev_arima_idx = previsao_arima.copy()
plt.plot(teste.index, prev_arima_idx.values,
         label=f'ARIMA (MAPE={mape_arima:.1f}%)',
         color='coral', linestyle='--', marker='o', markersize=4)

plt.plot(prev_teste_prophet['ds'].values, prev_teste_prophet['yhat'].values,
         label=f'Prophet (MAPE={mape_prophet:.1f}%)',
         color='seagreen', linestyle='--', marker='s', markersize=4)

plt.plot(datas_teste, prev_lstm,
         label=f'LSTM (MAPE={mape_lstm:.1f}%)',
         color='mediumpurple', linestyle='--', marker='^', markersize=4)

plt.axvline(datas_teste[0], color='gray', linestyle=':', linewidth=1, label='Início da previsão')
plt.title('Comparativo de Modelos — Previsão de MRR (últimos 12 meses)')
plt.ylabel('MRR (R$)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# comparativo de MAPE em barras
df_comparativo['MAPE (%)'].sort_values().plot(
    kind='barh', figsize=(8, 4),
    color=['seagreen', 'coral', 'mediumpurple'],
    edgecolor='white'
)
plt.title('MAPE por Modelo — Menor é melhor')
plt.xlabel('MAPE (%)')
plt.tight_layout()
plt.show()

# 9. Conclusão <a id="9"></a>

Nesta aula cobrimos três abordagens para forecasting de séries temporais aplicadas ao contexto de Telecom:

| Aspecto | ARIMA | Prophet | LSTM |
|---|---|---|---|
| **Complexidade** | Baixa | Baixa | Alta |
| **Dados necessários** | Poucos | Médio | Muitos |
| **Sazonalidade** | Manual | Automática | Aprende sozinho |
| **Interpretabilidade** | Alta | Alta | Baixa |
| **Feriados/Eventos** | Não | Sim (nativo) | Não (manual) |
| **Intervalo de confiança** | Sim | Sim | Não (padrão) |
| **Ideal para** | Séries curtas e estacionárias | Sazonalidade múltipla | Padrões não-lineares complexos |

**Conceitos essenciais desta aula:**

- Sempre decomponha a série antes de modelar — entender os componentes guia a escolha do modelo
- Teste a estacionariedade com o ADF antes de aplicar ARIMA
- No LSTM, a normalização dos dados é obrigatória
- MAPE é a métrica mais intuitiva para negócio — erro percentual médio
- Para horizontes curtos com dados limitados, ARIMA e Prophet costumam superar LSTM

Na próxima aula vamos trabalhar com **NLP Aplicado** — do TF-IDF ao BERT para análise de sentimentos e classificação de textos.

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)